# Classifier experiments v3: nine labels, and the perfect-evidence ceiling

Two questions, all runs BiomedBERT with `--class-weight sqrt`, 4 epochs:

- **E. Does training on the nine annotation labels help?** `--labels 9` against `--labels 3`,
  3 seeds each, both on our reranker's evidence. The 9-label model learns OVERSIMPLIFY,
  CONTRADICT, ETIQUETTE, ... separately, then its probabilities are summed into the three official
  groups, so both are scored the same way.
- **C. How good can the classifier get with perfect retrieval?** The same models trained and
  tested on top-20 documents that contain every gold evidence sentence (1 seed each). The gap
  between C and E is how much of the remaining error retrieval is responsible for.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow
Google Drive access. About 1.5 hours (8 runs of about 11 minutes). Each finished run is saved to
`MyDrive/CiteAudit_runs/v3`; if Colab disconnects, Run all again and finished runs are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/v3'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19"

import subprocess, json, statistics
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS', 'Loading weights')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
def train(name, labels, train_dir, dev_dir, tests, seed):
    out = f'{RES}/{name}_s{seed}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name} seed {seed}: already done'); return
    preds = ' '.join(f'--predict test={T}/{t}' for t in tests)
    run(f'python scripts/train_classifier.py --train {T}/{train_dir} --dev {T}/{dev_dir} '
        f'--labels {labels} --class-weight sqrt --seed {seed} --amp --eval-batch 16 '
        f'--out checkpoints/{name}_s{seed} --pred-dir {out}.part {preds}')
    run(f'cp checkpoints/{name}_s{seed}/config.json {out}.part/ && rm -rf checkpoints/{name}_s{seed}')
    os.rename(f'{out}.part', out)

## E. Nine labels against three, our reranker's evidence (about 65 minutes)

In [ ]:
for seed in (0, 1, 2):
    for labels in (9, 3):
        train(f'E{labels}', labels, 'ours_20_train_cv', 'ours_20', ['ours_20_test'], seed)

## C. Perfect-evidence ceiling (about 25 minutes)

Also scored on our real evidence, to see how a model trained on perfect evidence copes with
retrieval errors.

In [ ]:
for labels in (9, 3):
    train(f'C{labels}', labels, 'oracle_20_train', 'oracle_20', ['oracle_20_test', 'ours_20_test'], 0)

## Results

Mean ± sd over seeds. For reference, on test: the reproduced paper system has accuracy 0.584 and
macro-F1 0.524; our current best (MultiVerS + BiomedBERT ensemble) 0.627 and 0.536.

In [ ]:
LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(path):
    rows = [json.loads(l) for l in open(path)]
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1

print(f'{"run":5s} {"scored on":16s} ' + ' '.join(f'{c:>14s}' for c in ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']))
for name, files in [('E9', ['dev', 'ours_20_test']), ('E3', ['dev', 'ours_20_test']),
                    ('C9', ['dev', 'oracle_20_test', 'ours_20_test']), ('C3', ['dev', 'oracle_20_test', 'ours_20_test'])]:
    for f in files:
        runs = [scores(f'{RES}/{name}_s{s}/{f}.jsonl') for s in (0, 1, 2) if os.path.exists(f'{RES}/{name}_s{s}/{f}.jsonl')]
        if not runs: continue
        sd = [statistics.stdev(x) if len(runs) > 1 else 0. for x in zip(*runs)]
        print(f'{name:5s} {f:16s} ' + ' '.join(f'{statistics.mean(x):.3f} ± {s:.3f}' for x, s in zip(zip(*runs), sd)))

## Download

Zips `MyDrive/CiteAudit_runs/v3` (predictions and training logs, no weights). Put the unzipped
`v3` folder in the repository root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/v3.zip v3 -x '*.part/*'
from google.colab import files
files.download('/content/v3.zip')